# Clase 7: Redes neuronales. Laboratorio

Este notebook tiene dos tipos de secciones:

- **Secciones con código**: muestran cómo se usan en PyTorch las ideas de la clase. Ejecuta las celdas en orden y lee las explicaciones.
- **Ejercicios**: tienes que escribir código o experimentar. Después de cada ejercicio hay una **celda de verificación** que te dice si tu respuesta es correcta (✅) o qué está mal (❌). Puedes ver también pistas (▶ Pista).

| Ejercicio | Tema | Tiempo |
|---|---|---|
| 1 | La red de XOR con los pesos de las notas | ~10 min |
| 2 | La misma red sin ReLU | ~5 min |
| 3 | Contar parámetros | ~5 min |
| 4 | El gradiente con `backward()` | ~10 min |
| 5 | Entrenar una red | ~15 min |
| 6 | Explorar (opcional) | — |

Las partes 1 y 2 corresponden a las secciones 2 y 3 de las notas.

## Preparación

Esta celda importa las librerías y define algunas funciones para dibujar. No hace falta que la leas con detalle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import torch
from torch import nn
from sklearn.datasets import make_moons

C_NEG, C_POS = "#1f77b4", "#d62728"
FONDO = ListedColormap(["#d6e4f2", "#f6d5d5"])


def dibujar_puntos(ax, X, y, tam=30):
    X, y = np.asarray(X), np.asarray(y).ravel()
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c=C_NEG, marker="o", s=tam, label="clase 0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c=C_POS, marker="^", s=tam, label="clase 1")


def dibujar_frontera(ax, modelo, X, y, limites, n=300):
    """Pinta la clase que predice el modelo (salida > 0, es decir, probabilidad > 0.5)."""
    x0, x1, y0, y1 = limites
    xs, ys = np.meshgrid(np.linspace(x0, x1, n), np.linspace(y0, y1, n))
    P = torch.tensor(np.c_[xs.ravel(), ys.ravel()], dtype=torch.float32)
    with torch.no_grad():
        Z = (modelo(P) > 0).float().numpy().reshape(xs.shape)
    ax.contourf(xs, ys, Z, levels=[-0.5, 0.5, 1.5], cmap=FONDO)
    dibujar_puntos(ax, X, y)
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)


def verificar(condicion, ok, error):
    print(("✅ " + ok) if condicion else ("❌ " + error))


print("Listo. PyTorch", torch.__version__)

---
# Parte 1: La red como función

## 1.1 Tensores

En PyTorch los datos se guardan en **tensores**, que funcionan casi igual que los arreglos de NumPy. Igual que en scikit-learn, los ejemplos van en las **filas**: `X` es de $m \times n$ (un ejemplo por fila) y `y` es de $m \times 1$.

PyTorch trabaja por defecto con números `float32`, por eso escribimos `0.` en lugar de `0`.

In [ ]:
X_xor = torch.tensor([[0., 0.],
                      [0., 1.],
                      [1., 0.],
                      [1., 1.]])
y_xor = torch.tensor([[0.], [1.], [1.], [0.]])
print("X:", X_xor.shape, " y:", y_xor.shape)

## 1.2 Una neurona

Una capa densa (totalmente conectada) es `nn.Linear(n_entradas, n_salidas)`. Con `n_salidas = 1` es una sola neurona: calcula $z = \mathbf{w}^T\mathbf{x} + b$. Si le aplicamos la sigmoide, es exactamente la regresión logística de la clase 4.

PyTorch inicializa los pesos al azar. Para ver qué hace la neurona, le ponemos pesos a mano: con $\mathbf{w} = (1, 1)$ y $b = -1.5$, el score solo es positivo si las dos entradas son 1.

Fíjate en la forma de `weight`: es de **(salidas × entradas)**. Cada fila son los pesos de una neurona.

In [ ]:
neurona = nn.Linear(2, 1)
print("pesos:", neurona.weight.shape, " sesgo:", neurona.bias.shape)

with torch.no_grad():                      # no queremos que PyTorch registre esto para el gradiente
    neurona.weight.copy_(torch.tensor([[1., 1.]]))
    neurona.bias.copy_(torch.tensor([-1.5]))

z = neurona(X_xor)
print("scores z:", z.detach().ravel().tolist())
print("probabilidades:", [round(p, 2) for p in torch.sigmoid(z).ravel().tolist()])

Esta neurona calcula el AND (y lógico): solo el ejemplo $(1, 1)$ tiene probabilidad mayor que 0.5. Con otros pesos podríamos obtener el OR. Pero ningún par de pesos da el XOR, porque la frontera de una neurona es una recta.

---
# Ejercicio 1: La red de XOR

Construye la red de XOR de la **sección 2.1 de las notas** y comprueba que resuelve XOR. La red ya está creada con `nn.Sequential`, que aplica las capas en orden:

- `red_xor[0]`: la capa oculta, `nn.Linear(2, 2)`;
- `red_xor[1]`: la ReLU;
- `red_xor[2]`: la salida, `nn.Linear(2, 1)`, sin activación.

Tienes que asignar los pesos y sesgos de las notas: $a_1 = \text{ReLU}(x_1 + x_2)$, $a_2 = \text{ReLU}(x_1 + x_2 - 1)$, $\hat y = a_1 - 2a_2$.

<details><summary>▶ Pista</summary>

La matriz de pesos de la capa oculta es de 2 × 2: la fila 1 son los pesos de $a_1$ y la fila 2 los de $a_2$. El vector de sesgos tiene un número por neurona: `torch.tensor([0., -1.])`. La salida tiene una matriz de 1 × 2: `torch.tensor([[1., -2.]])`.
</details>

In [ ]:
red_xor = nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1))

with torch.no_grad():
    # TU CÓDIGO AQUÍ: completa los valores
    red_xor[0].weight.copy_(torch.tensor([[1., 1.], [1., 1.]]))
    red_xor[0].bias.copy_(torch.tensor([0., -1.]))
    red_xor[2].weight.copy_(torch.tensor([[1., -2.]]))
    red_xor[2].bias.copy_(torch.tensor([1.]))

print("y_hat:", red_xor(X_xor).detach().ravel().tolist())

In [ ]:
# Celda de verificación: no la modifiques
with torch.no_grad():
    h = red_xor[1](red_xor[0](X_xor))
    salida = red_xor(X_xor)
verificar(torch.allclose(h, torch.tensor([[0., 0.], [1., 0.], [1., 0.], [2., 1.]])),
          "la capa oculta da (a1, a2) = (0,0), (1,0), (1,0), (2,1), como en el Ejercicio 1 de las notas",
          f"las activaciones de la capa oculta deberían ser (0,0), (1,0), (1,0), (2,1), y son {h.tolist()}")
verificar(torch.allclose(salida, y_xor),
          "la red resuelve XOR: y_hat = 0, 1, 1, 0",
          f"y_hat debería ser 0, 1, 1, 0, y es {salida.ravel().tolist()}")

Compara con tu tabla del Ejercicio 1 de las notas. La celda siguiente dibuja la región donde la red predice $\hat y > 0.5$ (como en la sección 2.2 de las notas). Para dibujarla restamos 0.5 a la salida, porque `dibujar_frontera` pinta donde la salida es positiva.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
dibujar_frontera(ax, lambda P: red_xor(P) - 0.5, X_xor, y_xor, (-0.5, 1.5, -0.5, 1.5))
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.legend(loc="upper right")
plt.show()

---
# Ejercicio 2: Sin ReLU

Construye `red_lineal`, igual a `red_xor` pero **sin la ReLU**, y cópiale los mismos pesos y sesgos. Antes de ejecutar, escribe en papel qué salida esperas (es el Ejercicio 2 de las notas).

<details><summary>▶ Pista</summary>

`nn.Sequential(nn.Linear(2, 2), nn.Linear(2, 1))`. Para copiar los pesos: `red_lineal[0].weight.copy_(red_xor[0].weight)`, y lo mismo con el sesgo. Cuidado: en `red_lineal` la segunda capa es `red_lineal[1]`, no `red_lineal[2]`.
</details>

In [ ]:
# TU CÓDIGO AQUÍ
red_lineal = None

if red_lineal is not None:
    print("y_hat sin ReLU:", red_lineal(X_xor).detach().ravel().tolist())

In [ ]:
# Celda de verificación: no la modifiques
with torch.no_grad():
    salida = red_lineal(X_xor) if red_lineal is not None else torch.zeros(4, 1)
verificar(red_lineal is not None and len(red_lineal) == 2 and torch.allclose(salida, torch.tensor([[2.], [1.], [1.], [0.]])),
          "sin ReLU, y_hat = 2, 1, 1, 0 = 2 - x1 - x2: una función lineal que ya no resuelve XOR",
          f"y_hat debería ser 2, 1, 1, 0, y es {salida.ravel().tolist()}")

**Pregunta.** ¿Puedes escoger *otros* pesos para `red_lineal` que sí resuelvan XOR? ¿Por qué no? (Recuerda: sin activación, toda la red es una función lineal de $x_1$ y $x_2$.)

---
# Ejercicio 3: Contar parámetros

Construye la red para reconocer dígitos de la sección 2.5 de las notas: 784 entradas, dos capas ocultas de 16 neuronas con ReLU y una salida de 10 neuronas. Después cuenta sus parámetros.

`modelo.parameters()` recorre todos los tensores de pesos y sesgos del modelo, y `p.numel()` es el número de elementos de un tensor `p`.

<details><summary>▶ Pista</summary>

```python
red_mnist = nn.Sequential(
    nn.Linear(784, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 10),
)
```
Para contar: `sum(p.numel() for p in red_mnist.parameters())`.
</details>

In [ ]:
# TU CÓDIGO AQUÍ
red_mnist = nn.Sequential(nn.Linear(784, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 10))
n_parametros = None

print("Parámetros:", n_parametros)

In [ ]:
# Celda de verificación: no la modifiques
verificar(n_parametros == 13002, "13002 parámetros, igual que en las notas",
          f"deberían ser 13002 parámetros, y son {n_parametros}")
imagenes = torch.rand(5, 784)          # 5 "imágenes" al azar, una por fila
salida = red_mnist(imagenes) if red_mnist is not None else torch.zeros(5, 1)
verificar(tuple(salida.shape) == (5, 10), "la salida es de 5 × 10: un score por dígito para cada imagen",
          f"la salida debería ser de 5 × 10, y es {tuple(salida.shape)}")

Con softmax, los 10 scores de cada imagen se convierten en probabilidades que suman 1. Como la red no está entrenada, las probabilidades son parecidas entre sí: la red todavía no sabe nada.

In [ ]:
probs = torch.softmax(salida.detach(), dim=1)    # dim=1: softmax sobre cada fila
print("probabilidades de la primera imagen:", [round(p, 2) for p in probs[0].tolist()])
print("suma por fila:", [round(s, 4) for s in probs.sum(dim=1).tolist()])
print("dígito predicho para cada imagen:", probs.argmax(dim=1).tolist())

---
# Parte 2: Cómo aprende una red

## 2.1 Autograd

PyTorch puede calcular derivadas automáticamente. Si un tensor se crea con `requires_grad=True`, PyTorch registra todas las operaciones que se hacen con él. Al llamar `.backward()` sobre un resultado, recorre esas operaciones hacia atrás (backpropagation) y guarda la derivada de ese resultado respecto al tensor en el atributo `.grad`.

# Ejercicio 4: El gradiente con `backward()` (~10 min)

Repite el primer paso del Ejercicio 3 de las notas: la neurona $\hat y = w x$ con $w = 1$, $x = 2$, $y = 10$ y pérdida $L = (\hat y - y)^2$. Calcula `y_hat` y `L`, y llama `L.backward()`. A mano obtuvimos $dL/dw = 2(\hat y - y)x = -32$.

<details><summary>▶ Pista</summary>

`y_hat = w * x` y `L = (y_hat - y) ** 2`. Después, `L.backward()` y mira `w.grad`.
</details>

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
x, y = 2.0, 10.0

# TU CÓDIGO AQUÍ (3 líneas)
y_hat = None
L = None

print("L =", L if L is None else L.item(), "  dL/dw =", w.grad if w.grad is None else w.grad.item())

In [ ]:
# Celda de verificación: no la modifiques
verificar(L is not None and abs(L.item() - 64) < 1e-6, "L = 64",
          f"L debería ser 64, y es {L}")
verificar(w.grad is not None and abs(w.grad.item() + 32) < 1e-6,
          "dL/dw = -32: PyTorch calculó lo mismo que nosotros a mano",
          f"dL/dw debería ser -32, y es {w.grad} (¿llamaste L.backward()?)")

## 2.2 Descenso por gradiente a mano

Con `backward()` ya podemos hacer descenso por gradiente: calcular el gradiente, restarle $\alpha$ veces el gradiente al peso, y repetir. Dos detalles de PyTorch:

- La actualización se hace dentro de `torch.no_grad()`, porque no queremos que PyTorch la registre como parte del cálculo de la pérdida.
- PyTorch **suma** los gradientes de cada `backward()` a lo que ya hay en `.grad`. Por eso hay que borrarlo en cada paso con `w.grad.zero_()`.

Compara los primeros valores con tu tabla del Ejercicio 3 de las notas.

In [ ]:
def descenso(alfa, pasos=10):
    w = torch.tensor(1.0, requires_grad=True)
    valores = [w.item()]
    for _ in range(pasos):
        L = (w * 2.0 - 10.0) ** 2
        L.backward()
        with torch.no_grad():
            w -= alfa * w.grad
        w.grad.zero_()
        valores.append(w.item())
    return valores


print("alfa = 0.05:", [round(v, 3) for v in descenso(0.05)[:4]])

plt.figure(figsize=(6, 3.5))
for alfa in [0.05, 0.3]:
    plt.plot(descenso(alfa, 6), "o-", label=f"alfa = {alfa}")
plt.axhline(5, c="gray", ls=":")
plt.xlabel("paso"); plt.ylabel("w"); plt.legend()
plt.show()

**Pregunta.** Prueba `descenso(0.125, 3)`. ¿Qué pasa? ¿Y con `0.25`? ¿Puedes explicarlo? (Pista: el gradiente es $8(w - 5)$.)

## 2.3 Backpropagation en una red de dos pesos

Ahora comprobemos el ejemplo de la sección 3.3 de las notas: $a = \text{ReLU}(w_1 x)$, $\hat y = w_2 a$. A mano obtuvimos $\partial L/\partial w_1 = \partial L/\partial w_2 = -32$ con $w_1 = w_2 = 1$, y 0 para los dos con $w_1 = -1$ (la neurona está apagada).

In [ ]:
for w1_inicial in [1.0, -1.0]:
    w1 = torch.tensor(w1_inicial, requires_grad=True)
    w2 = torch.tensor(1.0, requires_grad=True)
    a = torch.relu(w1 * 2.0)
    y_hat = w2 * a
    L = (y_hat - 10.0) ** 2
    L.backward()
    print(f"w1 = {w1_inicial:4}:  a = {a.item()},  L = {L.item()},  dL/dw1 = {w1.grad.item() + 0.0},  dL/dw2 = {w2.grad.item() + 0.0}")

---
# Ejercicio 5: Entrenar una red

Ahora entrenamos una red para XOR **sin darle los pesos**: empieza con pesos al azar y el descenso por gradiente los encuentra. Usamos 8 neuronas ocultas en lugar de 2, porque con tan pocas neuronas a veces alguna queda "apagada" y el entrenamiento se atasca.

Completa las **5 líneas** del ciclo de entrenamiento (están en la sección 3.4 de las notas):

1. borrar los gradientes del paso anterior: `optimizador.zero_grad()`;
2. forward pass: `y_hat = modelo(X_xor)`;
3. pérdida: `L = perdida(y_hat, y_xor)`;
4. backpropagation: `L.backward()`;
5. un paso de descenso por gradiente: `optimizador.step()`.

La salida de la red es el score $z$, sin sigmoide: `nn.BCEWithLogitsLoss` aplica la sigmoide por nosotros. Por eso la red predice la clase 1 cuando su salida es mayor que 0.

<details><summary>▶ Pista</summary>

El orden importa: si llamas `step()` antes de `backward()`, no hay gradiente que usar; y si no llamas `zero_grad()`, los gradientes de todos los pasos se acumulan.
</details>

In [ ]:
torch.manual_seed(0)
modelo = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1))
perdida = nn.BCEWithLogitsLoss()
optimizador = torch.optim.SGD(modelo.parameters(), lr=0.1)

historia = []
for paso in range(2000):
    # TU CÓDIGO AQUÍ (5 líneas)
    L = torch.tensor(0.0)

    historia.append(L.item())

print("pérdida inicial:", round(historia[0], 3), "  pérdida final:", round(historia[-1], 4))

In [ ]:
# Celda de verificación: no la modifiques
with torch.no_grad():
    pred = (modelo(X_xor) > 0).float()
verificar(historia[-1] < historia[0], "la pérdida bajó durante el entrenamiento",
          "la pérdida no bajó: revisa el orden de las 5 líneas")
verificar(torch.equal(pred, y_xor), "la red aprendió XOR: acierta en los 4 ejemplos",
          f"la red predice {pred.ravel().tolist()} y debería predecir [0, 1, 1, 0]. "
          "Si la pérdida sí bajó, prueba otra semilla en torch.manual_seed o más pasos")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
axs[0].plot(historia)
axs[0].set_xlabel("paso"); axs[0].set_ylabel("pérdida"); axs[0].set_title("Pérdida de entrenamiento")
dibujar_frontera(axs[1], modelo, X_xor, y_xor, (-0.5, 1.5, -0.5, 1.5))
axs[1].set_title("Frontera aprendida")
plt.tight_layout()
plt.show()

La frontera aprendida no es igual a la franja de la sección 2.2 de las notas: hay muchas redes distintas que resuelven XOR, y el entrenamiento encuentra una de ellas.

---
# Ejercicio 6: Explorar (opcional)

La función `entrenar` entrena una red con una capa oculta sobre los datos de las **lunas** (los de la sección 2.7 de las notas) y dibuja la frontera. No hace falta modificarla: solo cambia los argumentos en la celda de abajo y observa qué pasa.

- `ocultas`: número de neuronas de la capa oculta;
- `relu`: si es `False`, la capa oculta no tiene activación;
- `lr`: tasa de aprendizaje;
- `pasos`: número de pasos de descenso por gradiente.

Responde:

1. ¿Cuántas neuronas ocultas hacen falta para que la frontera deje de ser una recta? ¿Y para separar bien las lunas? Con `ocultas=1` y la semilla 0, la red predice la misma clase para todos los puntos: ¿qué le pasó a esa neurona? (Revisa el Ejercicio 4 de las notas y prueba con `semilla=1`.)
2. Con `relu=False` y muchas neuronas, ¿cómo es la frontera? ¿Por qué?
3. ¿Qué pasa con la pérdida si usas `lr=5`? ¿Y con `lr=0.001`?

In [ ]:
X_lunas, y_lunas = make_moons(n_samples=200, noise=0.2, random_state=0)
X_lunas = torch.tensor(X_lunas, dtype=torch.float32)
y_lunas = torch.tensor(y_lunas, dtype=torch.float32).reshape(-1, 1)


def entrenar(ocultas=8, relu=True, lr=0.1, pasos=2000, semilla=0):
    torch.manual_seed(semilla)
    capas = [nn.Linear(2, ocultas)] + ([nn.ReLU()] if relu else []) + [nn.Linear(ocultas, 1)]
    modelo = nn.Sequential(*capas)
    perdida = nn.BCEWithLogitsLoss()
    optimizador = torch.optim.SGD(modelo.parameters(), lr=lr)
    historia = []
    for _ in range(pasos):
        optimizador.zero_grad()
        L = perdida(modelo(X_lunas), y_lunas)
        L.backward()
        optimizador.step()
        historia.append(L.item())

    with torch.no_grad():
        exactitud = ((modelo(X_lunas) > 0).float() == y_lunas).float().mean().item()
    n_par = sum(p.numel() for p in modelo.parameters())
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
    axs[0].plot(historia)
    axs[0].set_xlabel("paso"); axs[0].set_ylabel("pérdida")
    dibujar_frontera(axs[1], modelo, X_lunas, y_lunas, (-1.6, 2.6, -1.2, 1.7), n=200)
    axs[1].set_title(f"{ocultas} ocultas, relu={relu}: {n_par} parámetros, exactitud {exactitud:.0%}")
    plt.tight_layout()
    plt.show()

In [ ]:
entrenar(ocultas=1)

In [ ]:
# Prueba otros valores aquí
entrenar(ocultas=32, relu=True, lr=0.1, pasos=2000)